In [3]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import os
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter

def get_robust_session():
    session = requests.Session()
    retry = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=[500, 502, 503, 504],
        raise_on_status=False
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session

def scrape_goobjoog_article(url, session):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
    }
    
    try:
        response = session.get(url, headers=headers, timeout=20)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # 1. Extract Headline (Usually in h1 with entry-title class)
            headline_tag = soup.find('h1', class_='entry-title') or soup.find('h1')
            headline = headline_tag.get_text(strip=True) if headline_tag else "No Headline"
            
            # 2. Extract Body Content
            body_content = ""
            # Goobjoog typically uses 'entry-content' for the main text
            article_div = soup.find('div', class_='entry-content')
            
            if article_div:
                # Remove unwanted elements like social sharing or ads inside the content
                for extra in article_div(['script', 'style', 'aside', 'ins']):
                    extra.decompose()
                
                paragraphs = article_div.find_all('p')
                body_content = "\n".join([p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)])
            
            return headline, body_content
        else:
            return None, f"Status {response.status_code}"
    except Exception as e:
        return None, str(e)

# --- EXECUTION ---

# 1. Load your collected links
input_file = "goobjoog_ciyaaraha_links.xlsx"
output_file = "goobjoog_ciyaaraha_scraped_articles.xlsx"

if not os.path.exists(input_file):
    print(f"❌ Error: {input_file} not found. Run the link collector first.")
    exit()

df_links = pd.read_excel(input_file)
# If your column name in the Excel is "URL", ensure it matches here
urls_to_scrape = df_links['URL'].tolist() 

# 2. Setup Resume/Checkpoint Logic
scraped_data = []
processed_urls = set()

if os.path.exists(output_file):
    try:
        df_existing = pd.read_excel(output_file)
        scraped_data = df_existing.to_dict('records')
        processed_urls = set(df_existing['url'].tolist())
        print(f"🔄 Resuming: {len(processed_urls)} articles already scraped.")
    except:
        print("Starting fresh dataset.")

# 3. Main Loop
session = get_robust_session()
print(f"🚀 Starting scrape of {len(urls_to_scrape)} articles...")

for i, url in enumerate(urls_to_scrape):
    if url in processed_urls:
        continue
    
    print(f"[{i+1}/{len(urls_to_scrape)}] Scrapping: {url}")
    
    headline, body = scrape_goobjoog_article(url, session)
    
    # if headline and body and len(body) > 50:  # Ensure we actually got content
    scraped_data.append({
        'url': url,
        'headline': headline,
        'body': body,
        'category': 'cayaaraha',
        'source': 'goobjoog'
    })
    processed_urls.add(url)
    # else:
    #     print(f"   ⚠️ Skipping: {headline if headline else 'No content found'}")

    # Save every 5 articles to prevent data loss
    if (len(scraped_data)) % 5 == 0:
        pd.DataFrame(scraped_data).to_excel(output_file, index=False)
    
    # Crucial: Longer delay for Goobjoog to prevent 10054 errors
    time.sleep(2.5)

# Final Save
pd.DataFrame(scraped_data).to_excel(output_file, index=False)
print(f"\n✅ Scraping Complete! {len(scraped_data)} articles saved to {output_file}")

Starting fresh dataset.
🚀 Starting scrape of 674 articles...
[1/674] Scrapping: https://goobjoog.com/2020/12/31/yaa-guulaysan-doono-gobolka-banaadir-iyo-galmudug-tartanka-maamul-goboleedyada-2020/
[2/674] Scrapping: https://goobjoog.com/2020/10/25/xildhibaan-shaacir-guul-ayaan-u-rajaynaayaa-wasiir-xamsa-iyo-wasaarada-ciyaaraha-dalka/
[3/674] Scrapping: https://goobjoog.com/2024/01/21/akhriso-tartanka-dowlad-goboleedyada-oo-dib-u-bilaabanaya-ogow-isku-aadka-iyo-waxkasta-oo-ku-saabsan-ila-aiyo-final-ka/
[4/674] Scrapping: https://goobjoog.com/2020/07/11/maxamuud-maxamed-waxaan-diyaar-u-nahay-horomarka-iyo-kor-u-qaadida-ciyaaraha-degmada-diinsoor/
[5/674] Scrapping: https://goobjoog.com/2019/06/10/wax-badan-ka-ogaw-kulanka-horseed-vs-muqdisho-city-club/
[6/674] Scrapping: https://goobjoog.com/2020/06/16/ibraahim-geedow-xiriirka-waa-in-uu-wakiilo-u-diraa-xirfadlayaasha-jooga-kenya/
[7/674] Scrapping: https://goobjoog.com/2020/08/29/daadir-amiin-waan-ku-faraxsanahay-hanashada-horyaalka-heer

PermissionError: [Errno 13] Permission denied: 'goobjoog_ciyaaraha_scraped_articles.xlsx'